# MSUC validation tests

This notebook trains the SDDP policy on one instance, then simulates it. You choose the instance,
the horizon, the number of scenarios, the formulation (3-bin or interval) and the `train` settings
(number of iterations, optimality gap of the root MIP), then inspect the lower-bound trajectory,
the cost of the simulated policy and the breakdown of the computation time.

The rigorous checks (bound <= exact optimum <= policy, equivalence of the 3-bin and interval
formulations, Prop. 3) are in `examples/toy_convergence.jl` and `examples/compare_formulations.jl`.

Run from the repository root (the Julia kernel activates this project automatically).

## 1. Installing the dependencies (once)

Installs the packages listed in `Project.toml` (JuMP, Gurobi.jl, CSV, DataFrames, NCDatasets, ...).
Requires a valid Gurobi license on the machine.

In [ ]:
import Pkg
Pkg.activate(@__DIR__)  # activates this project, whichever Julia kernel is selected
Pkg.instantiate()

In [ ]:
using MSUC, Gurobi, Printf, Random

GRB_ENV = Gurobi.Env()
Gurobi.GRBsetintparam(GRB_ENV, "Threads", 1)

optimizer = () -> Gurobi.Optimizer(GRB_ENV)

In [ ]:
INSTANCE    = "Data/T-Ramp/10_0_5_w.nc4"  # path to the .nc4 file
T           = 24                           # horizon (number of time steps)
S           = 5                            # number of scenarios (support size of the ARMA innovation)
SIGMA       = 5.0                          # noise amplitude: multiplies the standard deviation of the ARMA innovation (0.0 = deterministic)
ITERATIONS  = 100                          # number of SDDP iterations
MIP_GAP     = 1e-3                         # root MIP optimality gap: 1e-2 = 1 %, 1e-3 = 0.1 %, nothing = solver default (1e-4)
PASSES_PER_MIP = 20                        # forward/backward (LP) passes per root MIP solve: 1 = usual, 5 or 10 = more cuts between two MIPs; ITERATIONS then counts cycles
PRINT_LEVEL = 1                            # 0 = silent, 1 = full MSUC.train log

# :threebin       -> 3-bin formulation (Sect. 2 of the paper, cuts (12))
# :extended       -> interval formulation (Sect. 3 of the paper, cuts (37)/(38))
# :threebin_gamma -> 3-bin formulation (same cuts) with the gamma variables added at the root (identical root relaxation)
METHOD = :extended

# Interval formulation only: linearization point ytilde^k of the non-selected intervals (Sect. 4.3), built in the forward pass
# :dual       -> price rule: sign of the cost signal z^F, rises fully if z^F < 0, falls fully if z^F > 0 (default)
# :projection -> unit output y projected onto the successors of ytilde_{t-1}
# :minimum    -> always the smallest successor
# :maximum    -> always the largest successor
INTERVAL_RULE = :dual

In [ ]:
FORMULATIONS = Dict(
    :threebin => (MSUC.subproblem_builder_UC_def,     MSUC.SDDPFormulation,     "3-bin"),
    :extended => (MSUC.subproblem_builder_UC_extended, MSUC.ExtendedFormulation, "interval"),
    :threebin_gamma => (MSUC.subproblem_builder_UC_def_gamma, MSUC.SDDPFormulation, "3-bin + gamma"),
)
builder, method, label = FORMULATIONS[METHOD]

instance = MSUC.parse_nc4(INSTANCE, optimizer, T; S = S, sigma = SIGMA)
@printf("instance %s: %d units, %d intervals, T = %d, S = %d, method = %s\n",
        basename(INSTANCE), instance.N,
        sum(length(u.intervals) for u in instance.Thermalunits), T, S, label)

In [ ]:
model = MSUC.PolicyGraph(
    builder, instance, 1.0, MSUC.Graph(T + 1), method;   # force = 1: the noise amplitude is set by SIGMA
    lower_bound = 0.0, optimizer = optimizer,
)

# Record the schedule x^k and the sampled states of each iteration without touching the algorithm
# (same records as `train_job` in runSMS.jl). With the interval formulation this includes the
# forward-pass interval state ytilde^k used by the diagnostics below.
record, record_pass, _, schedules, states = MSUC.make_recorder(model; interval_states = true)

Random.seed!(MSUC.SEED)   # same seed as `train_job` (runSMS.jl): runs are reproducible
MSUC.train(model; iteration_limit = ITERATIONS, mip_gap = MIP_GAP,
            passes_per_mip = PASSES_PER_MIP, interval_rule = INTERVAL_RULE,
            print_level = PRINT_LEVEL,
            forward_pass_callback = record_pass, post_iteration_callback = record)

In [ ]:
# Demand at bus b and stage t is Demandbus[b][t] * (1 + eta_t), where eta_t = S_eta zeta_t is the relative forecast
# error (common to all buses), zeta_t = Psi zeta_{t-1} + H xi_t (`arma.E` in the code), and the innovation xi_t takes S values:
# there are S^T possible paths, of which N_SCENARIOS are drawn at random.
using Plots, Statistics, Random, Printf

N_SCENARIOS = 200             # paths drawn from the model distribution
WITH_FORWARD_PASSES = true    # overlay the paths visited by the forward passes of the training

arma = instance.arma
forecast = [sum(instance.Demandbus[b][t] for b in eachindex(instance.Demandbus)) for t in 1:T]

function relative_error(innovations)
    zeta = copy(arma.zeta0)
    eta = zeros(length(innovations))
    for (t, xi) in enumerate(innovations)
        zeta = arma.Psi * zeta + arma.E * xi
        eta[t] = sum(arma.S_eta[1, :] .* zeta)
    end
    return eta
end

let
    rng = MersenneTwister(MSUC.SEED)
    cumulative = cumsum(arma.probability)
    draw() = arma.support[min(searchsortedfirst(cumulative, rand(rng)), length(cumulative))]
    eta = reduce(hcat, [relative_error([draw() for _ in 1:T]) for _ in 1:N_SCENARIOS])      # T x N_SCENARIOS

    # paths visited by the forward passes: zeta_t is the state at the exit of node t + 1
    visited = Matrix{Float64}(undef, T, 0)
    if WITH_FORWARD_PASSES && isdefined(Main, :states)
        trajectories = [vec(arma.S_eta[1:1, :] * reduce(hcat, p.sampled_states_zeta)[:, 2:end]) for k in eachindex(states) for p in states[k]]
        visited = reduce(hcat, trajectories)
    end

    # The scenarios come from the instance as built: if S or SIGMA changed since, rerun the instance and training cells.
    xs = [x[1] for x in arma.support]
    S_instance = length(xs)
    std_dev = sqrt(sum(arma.probability .* xs .^ 2))
    sigma_instance = std_dev / sqrt(MSUC.ARMA_INNOVATION_VARIANCE)
    @printf("innovation: %d points from %.2f %% to %.2f %% (standard deviation %.2f %%, i.e. sigma = %.2f); %d^%d possible paths, %d drawn",
            S_instance, 100 * minimum(xs), 100 * maximum(xs), 100 * std_dev, sigma_instance, S_instance, T, N_SCENARIOS)
    size(visited, 2) > 0 && @printf(", %d visited by the forward passes", size(visited, 2))
    println()
    S_instance == 1 && println("S = 1: the innovation is zero (a single point, 0), sigma has no effect")
    if S_instance != S || (S_instance > 1 && !isapprox(sigma_instance, SIGMA; atol = 1e-6))
        @warn "the instance was built with S = $S_instance and sigma = $(round(sigma_instance; digits = 3)), but S = $S and SIGMA = $SIGMA: rerun the instance cell (and the training cell)"
    end

    function panel(paths, nominal, visited_paths; ylabel, heading)
        lower = [quantile(view(paths, t, :), 0.05) for t in 1:T]
        upper = [quantile(view(paths, t, :), 0.95) for t in 1:T]
        p = plot(1:T, paths; color = :gray, alpha = 0.15, label = "", ylabel = ylabel, title = heading, titlefontsize = 10)
        size(visited_paths, 2) > 0 && plot!(p, 1:T, visited_paths; color = :darkorange, alpha = 0.3, label = "")
        plot!(p, 1:T, lower; fillrange = upper, fillalpha = 0.25, color = :royalblue, linewidth = 0, label = "5 % - 95 %")
        plot!(p, 1:T, vec(mean(paths; dims = 2)); color = :royalblue, linewidth = 2, label = "mean of the draws")
        plot!(p, 1:T, nominal; color = :black, linewidth = 2, linestyle = :dash, label = "forecast")
        size(visited_paths, 2) > 0 && plot!(p, [NaN], [NaN]; color = :darkorange, label = "forward passes")
        return p
    end
    p1 = panel(forecast .* (1 .+ eta), forecast, forecast .* (1 .+ visited); ylabel = "total demand (MW)", heading = "Demand scenarios")
    p2 = panel(100 .* eta, zeros(T), 100 .* visited; ylabel = "relative error (%)", heading = "Relative forecast error eta_t")
    plot(p1, p2; layout = (2, 1), size = (900, 650), xlabel = ["" "step t"], legend = [:topleft :none], left_margin = 6Plots.mm)
end

### Results: lower-bound trajectory and simulated policy cost

In [ ]:
using Plots
log = model.most_recent_training_results.log
for l in log
    @printf("it %3d   bound = %.2f   incumbent = %.2f\n", l.iteration, l.bound, l.incumbent)
end
plot([l.bound for l in log if l.iteration >= 50], title = "Lower bound", xlabel = "iteration", ylabel = "lower bound")

In [ ]:
t_sim = @elapsed sim = MSUC.simulate(model; seed = MSUC.SEED)   # same seed as `simulate_job` (runSMS.jl)
@printf("\nsimulated policy cost = %.2f  (%d paths, exhaustive = %s, %.1f s)\n",
        sim.cost, sim.npaths, sim.exhaustive, t_sim)
@printf("95 %% confidence interval: [%.2f, %.2f]  (standard error %.2f)\n",
        sim.ci95[1], sim.ci95[2], sim.stderr)
@printf("final bound = %.2f  ->  relative gap = %.2e\n",
        log[end].bound, 100*(sim.cost - log[end].bound) / abs(log[end].bound))

### Diagnostics: selected intervals and sampled states

`schedules[k]` is the schedule x^k chosen by the root MIP at iteration k (with the interval formulation,
its intervals are in `selected`) and `states[k][p]` is the trajectory of forward pass p of iteration k:
`sampled_states_power[n]` and `sampled_states_zeta[n]` are the state (output y, uncertainty state zeta) at the
exit of node n; node 1 is the initial state and node n is the state at the end of step n-1.

The first cell summarizes all iterations (bound, incumbent, new or already seen schedule, variation of the
sampled states); the second details one iteration by comparing it with the previous one; the third compares
y^k with y(gamma^k, ytilde^{k-1}) = sum, over the intervals selected at iteration k, of the linearization points
ytilde^{k-1}_{a,b,t} of the iteration k-1 cuts (built in the forward pass of iteration k-1: the visited output if the
interval was selected there, otherwise the minimum-cost trajectory under the prices of that pass): this is the output
that these cuts "expect" for schedule gamma^k. A gap means that the cut is evaluated at a point other than the one
where it was linearized.

In [ ]:
# Intervals (unit, a, b) of schedule x^k: the unit is on at steps a, ..., b-1 (b = T+1: until the end of the
# horizon; a <= 0: initial interval, the unit is already on at the start).
function selected_intervals(k)
    s, I = schedules[k], instance.intervals
    isempty(s.selected) || return [(I.unit[id], I.a[id], I.b[id]) for id in s.selected]
    # 3-bin: no interval is selected, recover the on-periods from u (column t+1 = step t)
    out = Tuple{Int,Int,Int}[]
    for i in 1:instance.N
        t = 0
        while t <= T
            if s.u[i, t+1] > 0.5
                a = t
                while t < T && s.u[i, t+2] > 0.5
                    t += 1
                end
                push!(out, (i, a, t + 1))
            end
            t += 1
        end
    end
    return out
end

function by_unit(k)
    d = [Tuple{Int,Int}[] for _ in 1:instance.N]
    for (i, a, b) in selected_intervals(k)
        push!(d[i], (a, b))
    end
    return sort!.(d)
end

# Output y (units x nodes) along forward pass p of iteration k
production(k, p = 1) = reduce(hcat, states[k][p].sampled_states_power)

# y(gamma^k, ytilde^{k-1}) = sum of gamma^k_{a,b} ytilde^{k-1}_{a,b,t}: the output that the iteration k-1 cuts
# "expect" when the schedule is gamma^k. ytilde^{k-1}_{a,b,t} is the point where the iteration k-1 cut was linearized
# for interval [a, b], built in the forward pass (price rule). Only the intervals active at step t+1 carry a y_t term
# in the cut; elsewhere the cut does not depend on y_t and `expected` is NaN. Column n = node n = step n-1.
# `is_new[i, n]`: the interval was not selected at iteration k-1.
function y_gamma(k)
    I = instance.intervals
    ytilde = states[k-1][end].sampled_states_interval    # last pass of iteration k-1 (most recent cut); ytilde[n][id] = interval id at the exit of node n
    already_selected = Set(schedules[k-1].selected)
    expected = fill(NaN, instance.N, T + 1)
    is_new = falses(instance.N, T + 1)
    for id in schedules[k].selected
        i, a, b = I.unit[id], I.a[id], I.b[id]
        for t in 0:(T-1)
            MSUC.is_on_interval(a, b, t + 1) || continue
            expected[i, t+1] = ytilde[t+1][id]
            is_new[i, t+1] = !(id in already_selected)
        end
    end
    return expected, is_new
end

# Largest absolute value, ignoring NaN
max_abs(M) = maximum(abs, filter(!isnan, M); init = 0.0)

K = length(log)
@printf("%4s %12s %12s %10s %6s %9s %10s %10s  %s\n",
        "it", "bound", "incumbent", "rel. gap", "#int.", "#changed", "max|dy|", "max|y-yg|", "schedule")
for k in 1:K
    n_changed = k == 1 ? "-" : string(count(by_unit(k) .!= by_unit(k - 1)))   # units whose intervals differ from k-1
    dy = k == 1 ? "-" : @sprintf("%.3g", maximum(abs, production(k) .- production(k - 1)))
    dyg = METHOD == :extended && k >= 2 ? @sprintf("%.3g", max_abs(production(k) .- y_gamma(k)[1])) : "-"   # y^k - y(gamma^k, ytilde^{k-1})
    seen = findfirst(j -> schedules[j].x == schedules[k].x, 1:k-1)
    @printf("%4d %12.2f %12.2f %10.2e %6d %9s %10s %10s  %s\n",
            k, log[k].bound, log[k].incumbent, (log[k].incumbent - log[k].bound) / abs(log[k].incumbent),
            length(selected_intervals(k)), n_changed, dy, dyg, seen === nothing ? "new" : "identical to iteration $seen")
end

In [ ]:
ITERATION = K-1    # iteration to examine, compared with the previous one
PASS = 1        # forward pass of that iteration (PASSES_PER_MIP passes per iteration)

current, previous = by_unit(ITERATION), by_unit(max(ITERATION - 1, 1))
interval_text(v) = isempty(v) ? "off" : join(["[$a, $b]" for (a, b) in v], " ")

println("Intervals of iteration $ITERATION; * : units that differ from iteration $(ITERATION - 1)\n")
for i in 1:instance.N
    isempty(current[i]) && isempty(previous[i]) && continue
    if current[i] == previous[i]
        println("   unit ", lpad(i, 3), " : ", interval_text(current[i]))
    else
        println(" * unit ", lpad(i, 3), " : ", interval_text(previous[i]), "  ->  ", interval_text(current[i]))
    end
end

y, y_prev = production(ITERATION, PASS), production(max(ITERATION - 1, 1), PASS)
zeta = reduce(hcat, states[ITERATION][PASS].sampled_states_zeta)    # zeta components x nodes
dy = y .- y_prev

println("\nSampled states of iteration $ITERATION (pass $PASS); zeta (components x nodes):")
display(round.(zeta; digits = 4))
println("\nUnits whose output y differs by at least 1e-6 from iteration $(ITERATION - 1):")
for i in findall(i -> maximum(abs, dy[i, :]) > 1e-6, 1:instance.N)
    n = argmax(abs.(dy[i, :]))
    @printf("   unit %3d : max gap %8.3f at node %2d  (y = %.3f vs %.3f)\n", i, abs(dy[i, n]), n, y[i, n], y_prev[i, n])
end

m = max(maximum(abs, dy), 1e-9)
heatmap(dy; xlabel = "node", ylabel = "unit", color = :balance, clims = (-m, m),
        title = "y(it $ITERATION) - y(it $(ITERATION - 1))")

In [ ]:
# y^k vs y(gamma^k, ytilde^{k-1}) at iteration ITERATION (defined in the previous cell)
if METHOD != :extended || ITERATION < 2
    println("comparison available with the interval formulation, from iteration 2")
else
    actual = production(ITERATION, PASS)
    expected, is_new = y_gamma(ITERATION)
    deviation = actual .- expected

    println("Units whose y^k differs from y(gamma^k, ytilde^{k-1}) by at least 1e-6 (iteration $ITERATION, pass $PASS)")
    println("(an interval is called new if it was not selected at iteration $(ITERATION - 1)):\n")
    for i in 1:instance.N
        valid = findall(!isnan, deviation[i, :])
        isempty(valid) && continue
        n = valid[argmax(abs.(deviation[i, valid]))]
        abs(deviation[i, n]) > 1e-6 || continue
        @printf("   unit %3d : max gap %8.3f at node %2d  (y = %.3f vs %.3f; interval %s)\n",
                i, abs(deviation[i, n]), n, actual[i, n], expected[i, n], is_new[i, n] ? "new" : "already selected")
    end
    @printf("\n%d (unit, node) pairs compared, %d with a new interval; max gap = %.3f\n",
            count(!isnan, deviation), count(is_new), max_abs(deviation))

    m = max(max_abs(deviation), 1e-9)
    heatmap(deviation; xlabel = "node", ylabel = "unit", color = :balance, clims = (-m, m),
            title = "y(it $ITERATION) - y(gamma^$ITERATION, ytilde^$(ITERATION - 1))")
end

In [ ]:
UNIT =  70   # unit to plot; the iteration and the pass are ITERATION and PASS of the detail cell

# Output trajectories of the unit: y^k (iteration k), y^{k-1} (iteration k-1, last pass: the one whose cuts are the most
# recent) and y(gamma^k, ytilde^{k-1}). Red diamonds: the interval was not selected at iteration k-1, so ytilde is the
# minimum-cost trajectory of the forward pass; gaps in the orange curve are steps where the cut does not depend on y_t.
# Step t is that of node n = t+1.
let
    unit, k = instance.Thermalunits[UNIT], ITERATION
    prev_k = max(k - 1, 1)
    y_k = production(k, PASS)[UNIT, :]
    y_prev = production(prev_k, length(states[prev_k]))[UNIT, :]
    expected, is_new = METHOD == :extended && k >= 2 ? y_gamma(k) : (fill(NaN, instance.N, T + 1), falses(instance.N, T + 1))
    describe_intervals(v) = isempty(v) ? "off" : join(["[$a, $b]" for (a, b) in v], " ")

    p = plot(0:T, y_prev; label = "y^(k-1)  (it $prev_k)", color = :lightgray, linewidth = 7,
             xlabel = "step t", ylabel = "output", legend = :outertopright, titlefontsize = 9, size = (900, 450),
             left_margin = 6Plots.mm, bottom_margin = 5Plots.mm,
             title = "unit $UNIT, iteration $k: $(describe_intervals(by_unit(k)[UNIT]))   (it $prev_k: $(describe_intervals(by_unit(prev_k)[UNIT])))")
    plot!(p, 0:T, y_k; label = "y^k  (it $k)", color = :royalblue, linewidth = 2, marker = :circle, markersize = 3)
    plot!(p, 0:T, expected[UNIT, :]; label = "y(gamma^k, ytilde^(k-1))", color = :darkorange,
          linewidth = 2, linestyle = :dash, marker = :circle, markersize = 3)
    scatter!(p, 0:T, [is_new[UNIT, n] ? expected[UNIT, n] : NaN for n in 1:T+1];
             label = "new interval", color = :red, marker = :diamond, markersize = 6)
    hline!(p, [unit.MaxPower]; label = "Pmax", color = :black, linestyle = :dot)
    hline!(p, [unit.MinPower]; label = "Pmin", color = :black, linestyle = :dot)
    p
end

In [ ]:
# Forward-pass cost signals z^F for unit UNIT (UNIT, ITERATION, PASS: previous cells). z^F_{j,a,b,t} (eq. (35c) with forward-pass
# duals) is the marginal cost of producing at step t, computed for all active intervals, selected or not. Two passes: iteration k-1
# (builds ytilde^{k-1}, the orange curve of the previous cell) and iteration k (PASS, produces y^k with the iteration k-1
# cut: it shows whether the rule was right). Rows: the unit's intervals selected at iteration k-1 (y^{k-1} follows them),
# then those selected only at iteration k (ytilde^{k-1} built by the rule). Top: sign of z^F at each active step, annotated
# with its value (blue: z^F < 0, a non-selected interval rises fully; red: z^F > 0, it falls fully; gray: |z^F| <= TOL_Z,
# indifferent, typically the marginal unit). Bottom: the two curves of the previous cell. Text: per interval, z^F of pass
# k-1, the lo/hi bounds on ytilde_t (successors of ytilde_{t-1}) and the resulting ytilde_t, to compare with y^{k-1}_t;
# then z^F of pass k and y^k_t. Step t is that of output y_t (node n = t+1).
TOL_Z = MSUC.SIGNAL_TOLERANCE     # |z^F| <= TOL_Z is treated as zero (same threshold as the forward-pass rule)
let
    k = ITERATION
    if METHOD != :extended || k < 2 || INTERVAL_RULE != :dual
        println("available with the interval formulation and INTERVAL_RULE = :dual (the only rule that reads the duals), from iteration 2")
    else
        unit, I = instance.Thermalunits[UNIT], instance.intervals
        yt = states[k-1][end].sampled_states_interval           # yt[n][id]: node n = step t+1 (yt[1]: initial state)
        Z = Dict(k - 1 => states[k-1][end].sampled_signals_interval,   # last pass of iteration k-1: builds ytilde^{k-1}
                 k => states[k][PASS].sampled_signals_interval)        # pass of iteration k: produces y^k; Z[p][n][id]
        prev_ids = [id for id in sort(schedules[k-1].selected) if I.unit[id] == UNIT]                          # y^{k-1} follows them
        only_k = [id for id in sort(schedules[k].selected) if I.unit[id] == UNIT && !(id in prev_ids)]         # ytilde^{k-1} from the rule
        ids = vcat(prev_ids, only_k)
        if isempty(ids)
            println("unit $UNIT: no interval selected at iteration $k nor at iteration $(k-1)")
        else
            steps = 1:T
            sign_class(v) = isnan(v) ? NaN : abs(v) <= TOL_Z ? 0.0 : sign(v)
            rows = [(p, id) for p in (k - 1, k) for id in ids]                      # (pass, interval), one row each
            classes = [sign_class(Z[p][t+1][id]) for (p, id) in rows, t in steps]   # NaN: inactive interval
            role(id) = id in prev_ids ? (id in schedules[k].selected ? "selected it $(k-1) and it $k" : "selected it $(k-1)") : "selected it $k only"
            labels = ["z it $p : [$(I.a[id]), $(I.b[id])]  " * role(id) for (p, id) in rows]
            y_prev = production(k - 1, length(states[k-1]))[UNIT, :]
            y_k = production(k, PASS)[UNIT, :]

            p1 = heatmap(steps, 1:length(rows), classes; color = cgrad([:royalblue, :lightgray, :tomato]), clims = (-1, 1), colorbar = false,
                         yflip = true, yticks = (1:length(rows), labels), xticks = steps, xlims = (-0.5, T + 0.5), ylims = (0.5, length(rows) + 0.5),
                         legend = :outertopright, titlefontsize = 9, title = "unit $UNIT: signs of z^F at the forward passes of iterations $(k-1) (builds ytilde^(k-1)) and $k (produces y^k)")
            annotate!(p1, [(t, r, text(@sprintf("%.3g", Z[p][t+1][id]), 6, :black)) for (r, (p, id)) in enumerate(rows) for t in steps if !isnan(Z[p][t+1][id])])
            hline!(p1, [length(ids) + 0.5]; color = :black, linewidth = 2, label = "")          # separates pass k-1 from pass k
            for (shade, caption) in ((:royalblue, "z^F < 0 : rises"), (:tomato, "z^F > 0 : falls"), (:lightgray, "|z^F| <= $TOL_Z : projection of y^k"))
                scatter!(p1, [NaN], [NaN]; color = shade, marker = :square, markersize = 6, markerstrokewidth = 0, label = caption)
            end

            direction(v) = abs(v) <= TOL_Z ? "~0" : v < 0 ? "rises" : "falls"
            for id in ids
                a, b = I.a[id], I.b[id]
                println("\n[$a, $b]  ", role(id), id in prev_ids ? " : ytilde = y^(k-1) (selected interval)" : " : ytilde built by the rule (interval not selected at iteration $(k-1))")
                println(" step   z^F it $(k-1)   sign   ytilde_(t-1)         lo         hi    ytilde_t   y^(k-1)_t   |   z^F it $k   sign       y^k_t")
                for t in steps
                    isnan(Z[k-1][t+1][id]) && continue
                    yt_prev = yt[t][id]
                    lo = max(MSUC.p_lower(unit, a, b, t), yt_prev - unit.DeltaRampDown)
                    hi = min(MSUC.p_upper(unit, a, b, t, T), yt_prev + MSUC.ramp_up(unit, a, b, t))
                    zk = Z[k][t+1][id]
                    @printf("%5d %11.3g %7s %14.2f %10.2f %10.2f %11.2f %11.2f   | %11.3g %7s %11.2f\n", t, Z[k-1][t+1][id], direction(Z[k-1][t+1][id]),
                            yt_prev, lo, hi, yt[t+1][id], y_prev[t+1], zk, isnan(zk) ? "-" : direction(zk), y_k[t+1])
                end
            end

            expected, is_new = y_gamma(k)
            p2 = plot(0:T, y_prev; label = "y^(k-1)  (it $(k-1))", color = :lightgray, linewidth = 7, xlims = (-0.5, T + 0.5),
                      xlabel = "step t", ylabel = "output", legend = :outertopright)
            plot!(p2, 0:T, y_k; label = "y^k  (it $k)", color = :royalblue, linewidth = 2, marker = :circle, markersize = 3)
            plot!(p2, 0:T, expected[UNIT, :]; label = "y(gamma^k, ytilde^(k-1))", color = :darkorange,
                  linewidth = 2, linestyle = :dash, marker = :circle, markersize = 3)
            scatter!(p2, 0:T, [is_new[UNIT, n] ? expected[UNIT, n] : NaN for n in 1:T+1];
                     label = "new interval", color = :red, marker = :diamond, markersize = 6)
            hline!(p2, [unit.MaxPower]; label = "Pmax", color = :black, linestyle = :dot)
            hline!(p2, [unit.MinPower]; label = "Pmin", color = :black, linestyle = :dot)
            plot(p1, p2; layout = @layout([a{0.38h}; b]), size = (900, 720), left_margin = 14Plots.mm, bottom_margin = 3Plots.mm)
        end
    end
end

In [ ]:
# Contribution of each unit to the change in value of the iteration k-1 cut when the visited state y^k departs from
# the point y(gamma^k, ytilde^{k-1}) where it was linearized. The cut is affine in y:
#     cut(gamma, y) = alpha + delta.zeta + sum, over the selected intervals, of (tau_id + beta_id y_unit),
# so cut(y^k) - cut(y(gamma^k, ytilde^{k-1})) = sum_j beta_j (y^k_j - ytilde_j), and each term is the contribution of
# unit j. Positive: the cut is higher (costlier future) at the visited point than at the linearization point; negative:
# lower. Sign of the slope beta_j: < 0, producing more today lowers the future cost (ramping up is valued); = 0, branch
# limited by capacity; > 0, producing costs. Node n is the state y_t with t = n-1; only nodes 2 to T carry a stage cut.
if METHOD != :extended || ITERATION < 2
    println("available with the interval formulation, from iteration 2")
else
    k, unit = ITERATION, instance.Thermalunits[UNIT]
    I = instance.intervals
    actual = production(k, PASS)
    expected, is_new = y_gamma(k)
    # cut of the last pass of iteration k-1: position 1 = trivial cut, then PASSES_PER_MIP per iteration
    position = 1 + (k - 1) * PASSES_PER_MIP

    # interval of gamma^k active at node n (step t+1 = n) for each unit
    active_id = zeros(Int, instance.N, T + 1)
    for id in schedules[k].selected, n in 1:T
        MSUC.is_on_interval(I.a[id], I.b[id], n) && (active_id[I.unit[id], n] = id)
    end

    slope = fill(NaN, instance.N, T + 1)     # beta_j of the cut
    contribution = fill(NaN, instance.N, T + 1)
    errors = Float64[]
    for n in 2:T
        cut = model[n].cuts[position]
        for i in 1:instance.N
            active_id[i, n] == 0 && continue
            slope[i, n] = cut.beta[active_id[i, n]]
            contribution[i, n] = slope[i, n] * (actual[i, n] - expected[i, n])
        end
        # check: sum of the contributions = cut(y^k) - cut(y(gamma^k, ytilde^{k-1}))
        zeta = states[k][PASS].sampled_states_zeta[n]
        cut_value(y) = cut.alpha + sum(cut.delta .* zeta) +
                    sum(cut.beta[id] * y[I.unit[id]] + cut.tau[id] for id in schedules[k].selected)
        y_tilde = ifelse.(isnan.(expected[:, n]), actual[:, n], expected[:, n])
        push!(errors, abs(cut_value(actual[:, n]) - cut_value(y_tilde) - sum(filter(!isnan, contribution[:, n]); init = 0.0)))
    end
    @printf("iteration %d, pass %d: cut of iteration %d (position %d). Gap between the sum of the contributions and the\n", k, PASS, k - 1, position)
    @printf("difference of the cut values, at the worst node: %.2e\n\n", maximum(errors))

    println("node  cut variation   of which positive   of which negative")
    for n in 2:T
        c = filter(!isnan, contribution[:, n])
        @printf("%5d %20.1f %15.1f %15.1f\n", n, sum(c; init = 0.0), sum(x -> max(x, 0.0), c; init = 0.0), sum(x -> min(x, 0.0), c; init = 0.0))
    end

    totals = [sum(filter(!isnan, contribution[i, :]); init = 0.0) for i in 1:instance.N]
    abs_totals = [sum(abs, filter(!isnan, contribution[i, :]); init = 0.0) for i in 1:instance.N]
    println("\nunits contributing the most (sum over the nodes, and sum of the absolute values):")
    for i in sortperm(abs_totals; rev = true)[1:min(10, instance.N)]
        @printf("   unit %3d : sum = %12.1f   sum of |.| = %12.1f\n", i, totals[i], abs_totals[i])
    end

    println("\nunit $UNIT: Pmin = $(unit.MinPower), Pmax = $(unit.MaxPower), max ramp-up R_up = $(unit.DeltaRampUp), max ramp-down R_dn = $(unit.DeltaRampDown), linear cost = $(unit.LinearTerm)")
    println("node   y^k    y~^(k-1)   gap    slope beta   contribution   rise y^k    interval")
    for n in 2:T
        id = active_id[UNIT, n]
        id == 0 && continue
        @printf("%5d %7.1f %9.1f %8.1f %12.3f %13.1f %12.1f    [%d, %d] %s\n", n, actual[UNIT, n], expected[UNIT, n], actual[UNIT, n] - expected[UNIT, n],
                slope[UNIT, n], contribution[UNIT, n], actual[UNIT, n] - actual[UNIT, n-1], I.a[id], I.b[id], is_new[UNIT, n] ? "(new)" : "")
    end

    m = max(maximum(abs, filter(!isnan, contribution); init = 0.0), 1e-9)
    heatmap(contribution; xlabel = "node", ylabel = "unit", color = :balance, clims = (-m, m),
            title = "contribution to cut(y^k) - cut(y(gamma^k, ytilde^(k-1))), it $k")
end

### Computation time by phase

Timers of `train`: root (MIP), forward pass, backward pass (LP solves, computation of the cut
coefficients, cut insertion) and schedule update.

In [ ]:
MSUC.TimerOutputs.print_timer(stdout, model.timer_output; compact = true)

### Root MIP time per iteration

The root problem is a MIP solved at every iteration: it is usually the dominant cost of `train`.
Its time grows with the number of accumulated cuts; the `MIP_GAP` option (parameters cell) can speed it up.

In [ ]:
using Plots
t_mip = model[1].computation_time[1:length(log)]   # one root MIP solve per iteration
@printf("root MIP: %.1f s in total, i.e. %.0f %% of the training time (%.1f s)\n",
        sum(t_mip), 100 * sum(t_mip) / log[end].time, log[end].time)
plot(t_mip, title = "Root MIP time", xlabel = "iteration", ylabel = "time (s)", legend = false)